**BOOKINGS**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import * 

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-7311041851026977>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import *
      3 from pyspark.sql.types import * 

ModuleNotFoundError: No module named 'dlt'

**Streaming Table** - Reads raw Bronze Delta table directly. No transformations — pure pass-through. Capture the Bronze data as a queryable streaming table

In [0]:
@dlt.table(
    name = "stage_bookings"
)
def stage_bokkings():

    df = spark.readStream.format("delta")\
        .load("/Volumes/workspace/bronze/bronzevolume/bookings/data")
    return df 

**Mat Views** - Reads from stage_bookings. Applies transformations (cast amount to double, add timestamp, convert date, drop rescue column)
Purpose: Transform the data (logic layer)

In [0]:
@dlt.view(
  name = "trans_bookings"
)
def trans_bookings():
  df = spark.readStream.table("stage_bookings")
  df = df.withColumn("amount", col("amount").cast("double")) \
  .withColumn("modifyDate", current_timestamp()) \
  .withColumn("booking_date", to_date(col("booking_date")))\
  .drop("_rescued_data")

return df 

**Rules**

In [0]:
rules = {
    "rule1" : "booking_id IS NOT NULL",
    "rule2" : "passenger_id IS NOT NULL"
}

**Streaming Table** - Reads from the trans_bookings view
Applies data quality rules (booking_id NOT NULL, passenger_id NOT NULL)
Purpose: Materialize cleaned, validated data for downstream use

In [0]:
@dlt.table(
    name = "silver_bookings"
)
@dlt.except_all_or_drop(rules)
def silver_bookings():
    df = spark.readStream.table("trans_bookings")

return df 